# 06 · Las 10 preguntas ciegas (día 24)

**Qué cubre:** el contrato del enunciado (§4.5) — `responder()` y `evaluar()` ejecutables sobre un clon limpio, sin tocar código — y el entregable de la presentación "el resultado de las 10 preguntas ciegas, con su delta contra el golden set propio".

**El día 24 hay 20 minutos** para ejecutar `holdout.jsonl` (10 preguntas, mismo esquema que el golden propio, al menos 2 sin respuesta en el corpus) y llevar el resultado a la presentación. Este notebook es exactamente lo que se ejecuta ese día: tres celdas de trabajo, nada que escribir a mano salvo el nombre del fichero.

**Antes del 23, ensayarlo es obligatorio** (lo pide el propio enunciado). Con `MODO_ENSAYO = True` (la celda siguiente) corre sobre `golden/oficial_20.jsonl` en vez de `holdout.jsonl`, que tiene el mismo esquema y sirve de sustituto — es del profesor, así que el resultado sí importa, pero no es la nota final.

In [ ]:
import os
import sys
import time
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
from IPython.display import Markdown, display

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente import evaluadores
from agente.agente import CONFIG_POR_DEFECTO
from agente.interfaz import evaluar

RES = RAIZ / "resultados"
print(f"Sistema que se va a evaluar (CONFIG_POR_DEFECTO): {CONFIG_POR_DEFECTO!r}")

## El fichero de preguntas

`MODO_ENSAYO = True` usa `golden/oficial_20.jsonl` como sustituto de `holdout.jsonl`; en clase se cambia a `False`. `RUTA_HOLDOUT` es el único nombre de fichero que hay que tocar si el profesor lo entrega con otro nombre.

In [ ]:
MODO_ENSAYO = True   # False el día 24, con holdout.jsonl ya en la raíz del repo
RUTA_HOLDOUT = RAIZ / "golden/oficial_20.jsonl" if MODO_ENSAYO else RAIZ / "holdout.jsonl"

if not RUTA_HOLDOUT.is_file():
    raise FileNotFoundError(
        f"No encuentro {RUTA_HOLDOUT}. Comprueba el nombre exacto del fichero "
        f"que ha entregado el profesor y actualiza RUTA_HOLDOUT.")

preguntas = evaluadores.cargar_golden(RUTA_HOLDOUT)
sin_campos = [p["id"] for p in preguntas if not {"id", "pregunta", "familia"} <= p.keys()]
assert not sin_campos, f"faltan campos básicos en: {sin_campos}"

print(f"{RUTA_HOLDOUT.name}: {len(preguntas)} preguntas")
familias = pd.Series([p["familia"] for p in preguntas]).value_counts()
display(familias.rename("preguntas"))
sin_cifra = sum(1 for p in preguntas if p["familia"] in ("numerica", "comparativa") and p.get("cifra_esperada") is None)
print(f"Preguntas sin cifra esperada (huecos del corpus): {sin_cifra}")

## Ejecutar

Una sola llamada: `evaluar()` ya imprime el progreso pregunta a pregunta, el resumen y el desglose por familia, y guarda el detalle en `resultados/eval_<etiqueta>.csv` — es el mismo código que se probó en los notebooks 03, 04 y 05, no algo nuevo para hoy.

Se cronometra aparte porque el límite son 20 minutos: si al terminar quedan menos de 5, hay que ir directamente a leer el resultado y dejar el resto de la sesión para la presentación.

In [ ]:
ETIQUETA = "ensayo_oficial" if MODO_ENSAYO else "holdout"

comienzo = time.time()
df = evaluar(str(RUTA_HOLDOUT), etiqueta=ETIQUETA)
minutos = (time.time() - comienzo) / 60

print(f"\nTiempo de la tirada: {minutos:.1f} min")
if not MODO_ENSAYO and minutos > 15:
    print("AVISO: quedan menos de 5 minutos de los 20 — ir directo a la lectura del resultado.")

## El delta contra el golden propio

Se compara con `resultados/eval_cifras_comparadas_propio.csv`, la tirada del sistema oficial sobre las 20 preguntas del golden propio (notebook 05) — es el número con el que se entrega el informe. **Si el hold-out sale peor, no es un suspenso: es el hallazgo que pide el enunciado.** Hay que poder decir qué parte de la mejora era general y cuál era memoria del conjunto con el que se iteró.

In [ ]:
propio = pd.read_csv(RES / "eval_cifras_comparadas_propio.csv")
resumen_propio = evaluadores.resumir(propio, "golden propio (informe)")
resumen_holdout = evaluadores.resumir(df, "oficial (ensayo)" if MODO_ENSAYO else "holdout (10 ciegas)")

comparacion = pd.DataFrame([resumen_propio, resumen_holdout]).set_index("sistema")
COLUMNAS = {"cita_ok": "cita", "cifra_ok": "cifra", "tool_ok": "trayectoria", "recall@5": "recall@5",
            "coste_medio_¢": "coste (¢)", "latencia_media_s": "latencia (s)", "errores": "errores"}
comparacion = comparacion[list(COLUMNAS)].rename(columns=COLUMNAS)

delta = comparacion.diff().iloc[-1].round(3)
comparacion.loc["delta"] = delta

display(comparacion.round(3))

nombre = "ensayo_vs_propio" if MODO_ENSAYO else "holdout_vs_propio"
comparacion.round(3).to_csv(RES / f"{nombre}.csv")
print(f"Guardado: resultados/{nombre}.csv")

## Para la presentación

- **La fila `delta`** de la tabla anterior es el número que va en la diapositiva: positivo es que el hold-out fue mejor que el golden propio, negativo que fue peor.
- **Antes de defenderlo, mira los errores** (`resultados/eval_<etiqueta>.csv`, columna `error`) y las preguntas concretas que fallaron: con solo 10 preguntas, una sola pesa un 10 % en cada métrica.
- **Al menos 2 de las 10 no tienen respuesta en el corpus.** Que el sistema conteste `fuente='ninguna'` sin cifra en esas es la mitad del examen, y se lee en la columna `cifra_ok` de esas filas.

In [ ]:
print("NOTEBOOK 06 COMPLETADO.")
print(f"Modo: {'ENSAYO (golden/oficial_20.jsonl)' if MODO_ENSAYO else 'HOLD-OUT REAL'}")
print(f"Resultado detallado: resultados/eval_{ETIQUETA}.csv")
print(f"Delta contra el golden propio: resultados/{'ensayo_vs_propio' if MODO_ENSAYO else 'holdout_vs_propio'}.csv")